In [66]:
with open("C:/Users/T Venkata Karthik/Downloads/sherlock.txt","r") as file:
    raw_data=file.read()

In [67]:
len(raw_data)

562202

In [68]:
import re
words = re.findall(r"\w+|[^\w\s]", raw_data)
words=list(set(words))
words.extend(["<unk>","<endOfText>"])
print(len(words))

8534


In [69]:
vocab={j:i for i,j in enumerate(words)}


TokenizerV1 verssion 1 uses words as tokens

In [70]:
class TokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:j for j,i in vocab.items()}
    def encode(self,text):
        words = re.findall(r"\w+|[^\w\s]", text)
        words = [word for word in words if word]
        l=[]
        for i in words:
            if i in self.str_to_int:
                l.append(self.str_to_int[i])
            else:
                l.append(self.str_to_int["<unk>"])
        return l
    def decode(self,nums):
        text = " ".join(self.int_to_str[i] for i in nums)
        text = re.sub(r"\s+([.,!?;:])", r"\1", text)
        return text

In [71]:
t=TokenizerV1(vocab)
s="His manner was not effusive. It seldom was; but he was glad, I think karthik"
encoded=t.encode(s)
print(encoded)

[5600, 887, 3833, 6565, 2343, 5183, 8529, 7404, 3833, 2042, 6689, 3393, 3833, 7767, 1103, 185, 6356, 8532]


In [72]:
decoded=t.decode(encoded)
print(decoded)

His manner was not effusive. It seldom was; but he was glad, I think <unk>


Now let us see TokenizerV2 version 2 which uses BPE (byte pair encoding)

In [73]:
d={} 
tokens = re.findall(r"\w+|\s+|[^\w\s]", raw_data)
for token in tokens:
    d[tuple(token)] = d.get(tuple(token), 0) + 1 
merge = sorted(d.items(), key=lambda x: x[1], reverse=True)
print(d)

{('T', 'h', 'e'): 341, (' ',): 95179, ('A', 'd', 'v', 'e', 'n', 't', 'u', 'r', 'e', 's'): 1, ('o', 'f'): 2625, ('S', 'h', 'e', 'r', 'l', 'o', 'c', 'k'): 95, ('H', 'o', 'l', 'm', 'e', 's'): 459, ('\n', '\n'): 2494, ('b', 'y'): 322, ('A', 'r', 't', 'h', 'u', 'r'): 21, ('C', 'o', 'n', 'a', 'n'): 1, ('D', 'o', 'y', 'l', 'e'): 1, ('\n', '\n', '\n'): 23, ('C', 'o', 'n', 't', 'e', 'n', 't', 's'): 1, ('\n', '\n', ' ', ' ', ' '): 1, ('I',): 3038, ('.',): 6208, (' ', ' ', ' ', ' ', ' '): 3, ('A',): 104, ('S', 'c', 'a', 'n', 'd', 'a', 'l'): 2, ('i', 'n'): 1692, ('B', 'o', 'h', 'e', 'm', 'i', 'a'): 11, ('\n', ' ', ' ', ' '): 11, ('I', 'I'): 3, (' ', ' ', ' ', ' '): 5, ('R', 'e', 'd'): 6, ('-',): 739, ('H', 'e', 'a', 'd', 'e', 'd'): 1, ('L', 'e', 'a', 'g', 'u', 'e'): 13, ('I', 'I', 'I'): 3, (' ', ' ', ' '): 3, ('C', 'a', 's', 'e'): 1, ('I', 'd', 'e', 'n', 't', 'i', 't', 'y'): 1, ('I', 'V'): 2, ('B', 'o', 's', 'c', 'o', 'm', 'b', 'e'): 15, ('V', 'a', 'l', 'l', 'e', 'y'): 6, ('M', 'y', 's', 't', 'e',

In [74]:

def mergePairs(vocab,d):
    merges=[]
    for _ in range(150):
        
        # Count pair frequencies
        pair_freq = {}
        for word, freq in d.items():
            for i in range(len(word)-1):
                pair = (word[i], word[i+1])
                pair_freq[pair] = pair_freq.get(pair, 0) + freq
        if not pair_freq:
            break
        # Find best pair
        best_pair = max(pair_freq, key=pair_freq.get)
    
        merges.append(best_pair)
        new_token = best_pair[0] + best_pair[1]
        vocab.add(new_token)
    
        # Merge it
        new_d = {}
    
        for word, freq in d.items():
            new_word = []
            i = 0
    
            while i < len(word):
                if (
                    i < len(word)-1 and
                    (word[i], word[i+1]) == best_pair
                ):
                    new_word.append(word[i] + word[i+1])
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
    
            new_word = tuple(new_word)
            new_d[new_word] = new_d.get(new_word, 0) + freq
    
        d = new_d
    return merges,vocab,d
   

In [75]:
# Initial vocabulary (before the loop)
vocab = set()
for word in d:
    vocab.update(word)
merges,vocab,d=mergePairs(vocab,d)


In [76]:
vocab={i:j for j,i in enumerate(sorted(vocab))}
if "<unk>" not in vocab:
    vocab["<unk>"] = len(vocab)
print(d)

{('The',): 341, (' ',): 95179, ('A', 'd', 'v', 'ent', 'u', 're', 's'): 1, ('of',): 2625, ('S', 'her', 'lo', 'ck'): 95, ('Holmes',): 459, ('\n\n',): 2494, ('by',): 322, ('A', 'r', 'th', 'ur'): 21, ('C', 'on', 'an'): 1, ('D', 'o', 'y', 'le'): 1, ('\n\n', '\n'): 23, ('C', 'on', 't', 'ent', 's'): 1, ('\n\n', ' ', ' ', ' '): 1, ('I',): 3038, ('.',): 6208, (' ', ' ', ' ', ' ', ' '): 3, ('A',): 104, ('S', 'c', 'and', 'al'): 2, ('in',): 1692, ('B', 'o', 'he', 'm', 'i', 'a'): 11, ('\n', ' ', ' ', ' '): 11, ('I', 'I'): 3, (' ', ' ', ' ', ' '): 5, ('R', 'ed'): 6, ('-',): 739, ('H', 'e', 'ad', 'ed'): 1, ('L', 'e', 'ag', 'u', 'e'): 13, ('I', 'I', 'I'): 3, (' ', ' ', ' '): 3, ('C', 'as', 'e'): 1, ('I', 'd', 'ent', 'it', 'y'): 1, ('I', 'V'): 2, ('B', 'o', 's', 'co', 'm', 'be'): 15, ('V', 'all', 'e', 'y'): 6, ('M', 'y', 'st', 'er', 'y'): 1, ('V',): 2, ('F', 'i', 've'): 3, ('O', 'r', 'an', 'ge'): 1, ('P', 'i', 'p', 's'): 1, ('V', 'I'): 2, ('M', 'an'): 2, ('with',): 803, ('the',): 5263, ('T', 'w', 'is',

In [77]:
final_vocab = set()

for word in d:
    final_vocab.update(word)

final_vocab.add("<unk>")

vocab = {tok: idx for idx, tok in enumerate(sorted(final_vocab))}
print(vocab)

{'\n': 0, '\n\n': 1, ' ': 2, '!': 3, '&': 4, '(': 5, ')': 6, ',': 7, '-': 8, '.': 9, '0': 10, '1': 11, '2': 12, '3': 13, '4': 14, '5': 15, '6': 16, '7': 17, '8': 18, '9': 19, ':': 20, ';': 21, '<unk>': 22, '?': 23, 'A': 24, 'B': 25, 'C': 26, 'D': 27, 'E': 28, 'F': 29, 'G': 30, 'H': 31, 'Hol': 32, 'Holmes': 33, 'I': 34, 'It': 35, 'J': 36, 'K': 37, 'L': 38, 'M': 39, 'N': 40, 'O': 41, 'P': 42, 'Q': 43, 'R': 44, 'S': 45, 'T': 46, 'Th': 47, 'The': 48, 'U': 49, 'V': 50, 'W': 51, 'X': 52, 'Y': 53, 'Z': 54, '_': 55, 'a': 56, 'ab': 57, 'ac': 58, 'ad': 59, 'ag': 60, 'ain': 61, 'al': 62, 'all': 63, 'am': 64, 'an': 65, 'and': 66, 'ap': 67, 'ar': 68, 'ard': 69, 'are': 70, 'as': 71, 'at': 72, 'ay': 73, 'b': 74, 'be': 75, 'been': 76, 'but': 77, 'by': 78, 'c': 79, 'ce': 80, 'ch': 81, 'ck': 82, 'co': 83, 'con': 84, 'ct': 85, 'd': 86, 'de': 87, 'do': 88, 'e': 89, 'ed': 90, 'en': 91, 'ent': 92, 'er': 93, 'ere': 94, 'es': 95, 'ess': 96, 'est': 97, 'et': 98, 'ex': 99, 'f': 100, 'fe': 101, 'fo': 102, 'for':

In [78]:
merge_rank={}
for rank, pair in enumerate(merges):
    merge_rank[pair] = rank
print(merge_rank)

{('t', 'h'): 0, ('i', 'n'): 1, ('e', 'r'): 2, ('a', 'n'): 3, ('th', 'e'): 4, ('o', 'u'): 5, ('a', 't'): 6, ('o', 'n'): 7, ('e', 'd'): 8, ('i', 's'): 9, ('e', 'n'): 10, ('a', 's'): 11, ('t', 'o'): 12, ('r', 'e'): 13, ('an', 'd'): 14, ('o', 'f'): 15, ('e', 's'): 16, ('in', 'g'): 17, ('h', 'a'): 18, ('h', 'e'): 19, ('i', 't'): 20, ('\n', '\n'): 21, ('o', 'r'): 22, ('a', 'r'): 23, ('l', 'l'): 24, ('m', 'e'): 25, ('w', 'h'): 26, ('l', 'e'): 27, ('s', 't'): 28, ('s', 'e'): 29, ('b', 'e'): 30, ('y', 'ou'): 31, ('o', 'w'): 32, ('v', 'e'): 33, ('th', 'at'): 34, ('c', 'h'): 35, ('w', 'as'): 36, ('r', 'o'): 37, ('i', 'd'): 38, ('w', 'i'): 39, ('l', 'y'): 40, ('l', 'd'): 41, ('n', 'o'): 42, ('v', 'er'): 43, ('c', 'e'): 44, ('g', 'h'): 45, ('er', 'e'): 46, ('h', 'is'): 47, ('u', 't'): 48, ('en', 't'): 49, ('s', 'o'): 50, ('m', 'y'): 51, ('a', 'y'): 52, ('a', 'd'): 53, ('r', 'i'): 54, ('a', 'l'): 55, ('f', 'or'): 56, ('wi', 'th'): 57, ('i', 'm'): 58, ('ha', 've'): 59, ('t', 'er'): 60, ('u', 'p'): 61

In [79]:
class TokenizerV2:
    def __init__(self, vocab, merge_rank):
        self.str_to_int = vocab
        self.int_to_str = {v: k for k, v in vocab.items()}
        self.merge_rank = merge_rank

    def bpe_encode(self, word):
        word = list(word)

        while len(word) > 1:
            best_pair = None
            best_rank = float("inf")

            # Find the mergeable pair with the smallest rank
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                if pair in self.merge_rank and self.merge_rank[pair] < best_rank:
                    best_rank = self.merge_rank[pair]
                    best_pair = pair

            if best_pair is None:
                break

            # Merge every occurrence of that pair
            new_word = []
            i = 0

            while i < len(word):
                if (
                    i < len(word) - 1
                    and (word[i], word[i + 1]) == best_pair
                ):
                    new_word.append(word[i] + word[i + 1])
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1

            word = new_word

        return word

    def encode(self, text):
        tokens = re.findall(r"\w+|\s+|[^\w\s]", text)

        ids = []

        for token in tokens:
            pieces = self.bpe_encode(token)

            for piece in pieces:
                ids.append(
                    self.str_to_int.get(piece, self.str_to_int["<unk>"])
                )

        return ids

    def decode(self, ids):
        return "".join(self.int_to_str[i] for i in ids)

In [80]:
tokenizer = TokenizerV2(vocab, merge_rank)

In [81]:
encoded=tokenizer.encode("I did so, Karthik and saw a large “E” with a small “g,” a “P,” and a large Hemanth “G” with a small “t” woven into the texture of the paper.")

In [82]:
print(encoded)

[34, 2, 86, 119, 2, 183, 7, 2, 37, 68, 190, 118, 132, 2, 66, 2, 177, 211, 2, 56, 2, 134, 68, 106, 2, 237, 28, 238, 2, 220, 2, 56, 2, 176, 141, 63, 2, 237, 105, 7, 238, 2, 56, 2, 237, 42, 7, 238, 2, 66, 2, 56, 2, 134, 68, 106, 2, 31, 89, 142, 190, 2, 237, 30, 238, 2, 220, 2, 56, 2, 176, 141, 63, 2, 237, 188, 238, 2, 211, 153, 207, 91, 2, 123, 197, 2, 192, 2, 188, 99, 188, 199, 172, 2, 155, 2, 192, 2, 166, 67, 93, 9]


In [83]:
decoded=tokenizer.decode(encoded)
print(decoded)

I did so, Karthik and saw a large “E” with a small “g,” a “P,” and a large Hemanth “G” with a small “t” woven into the texture of the paper.


In [84]:
encoded_text=tokenizer.encode(raw_data)
print(encoded_text)

[48, 2, 24, 86, 207, 92, 199, 172, 176, 2, 155, 2, 45, 115, 139, 82, 2, 33, 1, 78, 2, 24, 171, 190, 203, 2, 26, 157, 65, 2, 27, 153, 222, 136, 1, 0, 26, 157, 188, 92, 176, 1, 2, 2, 2, 34, 9, 2, 2, 2, 2, 2, 24, 2, 45, 79, 66, 62, 2, 123, 2, 25, 153, 114, 141, 118, 56, 0, 2, 2, 2, 34, 34, 9, 2, 2, 2, 2, 48, 2, 44, 90, 8, 31, 89, 59, 90, 2, 38, 89, 60, 199, 89, 0, 2, 2, 2, 34, 34, 34, 9, 2, 2, 2, 24, 2, 26, 71, 89, 2, 155, 2, 34, 86, 92, 129, 222, 0, 2, 2, 2, 34, 50, 9, 2, 2, 2, 2, 48, 2, 25, 153, 176, 83, 141, 75, 2, 50, 63, 89, 222, 2, 39, 222, 186, 93, 222, 0, 2, 2, 2, 50, 9, 2, 2, 2, 2, 2, 48, 2, 29, 118, 208, 2, 41, 171, 65, 106, 2, 42, 118, 166, 176, 0, 2, 2, 2, 50, 34, 9, 2, 2, 2, 2, 48, 2, 39, 65, 2, 220, 2, 192, 2, 46, 211, 128, 188, 90, 2, 38, 118, 166, 0, 2, 2, 2, 50, 34, 34, 9, 2, 2, 2, 48, 2, 24, 86, 207, 92, 199, 172, 2, 155, 2, 192, 2, 25, 134, 199, 89, 2, 26, 68, 74, 200, 79, 136, 0, 2, 2, 2, 50, 34, 34, 34, 9, 2, 2, 48, 2, 24, 86, 207, 92, 199, 172, 2, 155, 2, 192, 2, 45,

In [85]:
class DataSet:
    def __init__(self,encoded_text,length,stride=1):
        if not isinstance(stride, int) or stride < 1:
            stride = 1
        self.inputs=[]
        self.outputs=[]
        for i in range(0,len(encoded_text)-length-1,stride):
            self.inputs.append(encoded_text[i:i+length])
            self.outputs.append(encoded_text[i+1:i+length+1])
    def length(self):
        return len(self.inputs)
    def get(self,idx):
        return self.inputs[idx],self.outputs[idx]

In [86]:
setter=DataSet(encoded_text,16,16)
print(setter.get(1300))

([2, 192, 2, 83, 141, 125, 0, 155, 2, 129, 176, 2, 153, 79, 79, 201], [192, 2, 83, 141, 125, 0, 155, 2, 129, 176, 2, 153, 79, 79, 201, 65])


In [87]:
class DataLoader:
    def __init__(self,raw_data,batch_size,length,stride=16):
        if not isinstance(stride, int) or stride < 1:
            stride = 1
        if not isinstance(batch_size, int) or batch_size < 1:
            batch_size = 1
        if not isinstance(length, int) or length < 4:
            length=4
        encoded_text = tokenizer.encode(raw_data)
        self.dataset = DataSet(encoded_text, length, stride)
        self.batch_size = batch_size
        self.idx = 0
    def get_batch(self):
        inputs = []
        outputs = []

        for _ in range(self.batch_size):
            if self.idx >= self.dataset.length():
                break

            x, y = self.dataset.get(self.idx)
            inputs.append(x)
            outputs.append(y)
            self.idx += 1
        if len(inputs) < self.batch_size:
                return None, None

        return inputs, outputs
    def reset(self):
        self.idx = 0
        
        

In [90]:
import numpy as np
vocab_size=len(vocab)
dim=256
embedding_matrix=np.random.randn(vocab_size, 256) * 0.02
print(embedding_matrix)

[[-0.02693654  0.01340297  0.02166183 ... -0.0080375  -0.01262025
  -0.00264109]
 [ 0.01875629 -0.00079346 -0.00963531 ...  0.01871728  0.02582557
  -0.00037265]
 [ 0.01843691 -0.03252394  0.01036639 ... -0.00745998  0.01474909
   0.01034148]
 ...
 [ 0.01282039  0.00322602  0.01613945 ...  0.02263853  0.00990526
   0.01418801]
 [ 0.02442208  0.00715171 -0.01133555 ...  0.05155396 -0.01697136
  -0.02306525]
 [-0.01067471 -0.0320691   0.03317557 ... -0.01602668  0.01194487
  -0.01312223]]


In [91]:
'''vector_inputs=[]
for i in embedding_inputs:
    lst=[]
    for j in i:
        lst.append(embedding_matrix[j])
    vector_inputs.append(lst)
print(len(vector_inputs),len(vector_inputs[0]),len(vector_inputs[0][0]))
print(vector_inputs[0])'''

'vector_inputs=[]\nfor i in embedding_inputs:\n    lst=[]\n    for j in i:\n        lst.append(embedding_matrix[j])\n    vector_inputs.append(lst)\nprint(len(vector_inputs),len(vector_inputs[0]),len(vector_inputs[0][0]))\nprint(vector_inputs[0])'

In [92]:
import numpy as np

sequence_length = 16    # your context length
embedding_dim = 256

position_embeddings = np.random.randn(sequence_length, embedding_dim) * 0.02

print(position_embeddings.shape)
print(position_embeddings)

(16, 256)
[[-5.01175396e-02 -2.47394812e-02  9.88321301e-03 ... -2.02163654e-03
   4.05125160e-03  9.94169885e-04]
 [ 2.14895979e-03 -1.71144543e-02  5.83540060e-03 ... -4.07353703e-03
  -1.08228474e-02  1.11527858e-02]
 [-8.07088897e-03  3.37657809e-02 -7.65377906e-03 ... -4.15217096e-03
   1.05510948e-02 -8.83615139e-03]
 ...
 [-6.53058719e-03  2.56898002e-02  2.08378395e-02 ...  1.14263382e-02
   5.45679158e-03  4.07772824e-03]
 [-3.21223162e-03 -1.10699186e-02  1.71588999e-02 ...  9.47559592e-03
  -1.09979930e-03  3.27417754e-03]
 [-6.14910701e-05  1.16509291e-02 -1.73509881e-02 ...  1.70301022e-03
   6.92554067e-03 -3.93472823e-03]]


In [93]:
'''final_inputs = []

for batch in vector_inputs:
    batch_with_position = []

    for sequence in batch:
        # sequence shape = (4,256)
        sequence = sequence + position_embeddings
        batch_with_position.append(sequence)

    final_inputs.append(batch_with_position)
print(final_inputs[0])'''

'final_inputs = []\n\nfor batch in vector_inputs:\n    batch_with_position = []\n\n    for sequence in batch:\n        # sequence shape = (4,256)\n        sequence = sequence + position_embeddings\n        batch_with_position.append(sequence)\n\n    final_inputs.append(batch_with_position)\nprint(final_inputs[0])'

In [94]:
"""attention=[]
for seq in final_inputs[0]:
    scores=seq@seq.T
    scores=scores-np.max(scores, axis=1, keepdims=True)
    exp=np.exp(scores)
    a=exp/np.sum(exp, axis=1, keepdims=True)
    attention.append(a)
    print(np.sum(a,axis=1))
print(attention)"""

'attention=[]\nfor seq in final_inputs[0]:\n    scores=seq@seq.T\n    scores=scores-np.max(scores, axis=1, keepdims=True)\n    exp=np.exp(scores)\n    a=exp/np.sum(exp, axis=1, keepdims=True)\n    attention.append(a)\n    print(np.sum(a,axis=1))\nprint(attention)'

In [95]:
"""outputs = []

for a, seq in zip(attention, final_inputs[0]):
    context = a @ seq
    outputs.append(context)

outputs = np.array(outputs)
print(outputs)"""

'outputs = []\n\nfor a, seq in zip(attention, final_inputs[0]):\n    context = a @ seq\n    outputs.append(context)\n\noutputs = np.array(outputs)\nprint(outputs)'

In [96]:
class SelfAttentionHead:

    def __init__(self, embed_dim, head_dim, sequence_length):

        self.embed_dim = embed_dim
        self.head_dim = head_dim

        self.Wq = np.random.randn(
            embed_dim,
            head_dim
        ) * 0.02

        self.Wk = np.random.randn(
            embed_dim,
            head_dim
        ) * 0.02

        self.Wv = np.random.randn(
            embed_dim,
            head_dim
        ) * 0.02

        # Causal mask
        self.mask = np.triu(
            np.ones(
                (sequence_length, sequence_length),
                dtype=bool
            ),
            k=1
        )

        self.cache = []

    def forward(self, seq):

        Q = seq @ self.Wq
        K = seq @ self.Wk
        V = seq @ self.Wv
    
        # ==========================================
        # Attention scores
        # ==========================================
    
        scale = np.sqrt(self.head_dim)
    
        scores = (Q @ K.T) / scale
    
        # ==========================================
        # Dynamic causal mask
        # ==========================================
    
        seq_len = seq.shape[0]
    
        mask = self.mask[:seq_len, :seq_len]
    
        scores[mask] = -1e9
    
        # ==========================================
        # Stable softmax
        # ==========================================
    
        scores = (
            scores
            - np.max(
                scores,
                axis=1,
                keepdims=True
            )
        )
    
        exp_scores = np.exp(scores)
    
        attn = (
            exp_scores
            / np.sum(
                exp_scores,
                axis=1,
                keepdims=True
            )
        )
    
        # ==========================================
        # Context
        # ==========================================
    
        context = attn @ V
    
        # ==========================================
        # Cache
        # ==========================================
    
        self.cache.append(
            (
                seq,
                Q,
                K,
                V,
                attn
            )
        )
    
        return context

    def backward(self, dcontext, learning_rate=1e-3):

        # Because we use pop(),
        # TransformerBlock.backward()
        # must process sequences in reverse order.

        seq, Q, K, V, attn = self.cache.pop()

        scale = np.sqrt(self.head_dim)

        # ============================
        # Context = attn @ V
        # ============================

        d_attn = dcontext @ V.T

        dV = attn.T @ dcontext

        # ============================
        # V weights
        # ============================

        dWv = seq.T @ dV

        dx_v = dV @ self.Wv.T

        # ============================
        # Softmax backward
        # ============================

        d_scores = np.zeros_like(attn)

        for i in range(attn.shape[0]):

            a = attn[i]
            da = d_attn[i]

            J = np.diag(a) - np.outer(a, a)

            d_scores[i] = J @ da

        # Do not propagate gradients
        # through masked positions
        d_scores[self.mask] = 0

        # ============================
        # Q and K
        # ============================

        dQ = (d_scores @ K) / scale

        dK = (d_scores.T @ Q) / scale

        # ============================
        # Q/K weights
        # ============================

        dWq = seq.T @ dQ
        dWk = seq.T @ dK

        dx_q = dQ @ self.Wq.T
        dx_k = dK @ self.Wk.T

        # ============================
        # Update weights
        # ============================

        self.Wq -= learning_rate * dWq

        self.Wk -= learning_rate * dWk

        self.Wv -= learning_rate * dWv

        # ============================
        # Gradient to input
        # ============================

        dx = dx_q + dx_k + dx_v

        return dx

In [97]:
class MultiHeadAttention:

    def __init__(
        self,
        embed_dim=256,
        num_heads=4,
        sequence_length=16
    ):

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.heads = [
            SelfAttentionHead(
                embed_dim,
                self.head_dim,
                sequence_length
            )
            for _ in range(num_heads)
        ]

        # Output projection
        self.Wo = np.random.randn(
            embed_dim,
            embed_dim
        ) * 0.02

        self.cache = []

    def forward(self, seq):

        head_outputs = []

        # ============================
        # Run every attention head
        # ============================

        for head in self.heads:

            head_outputs.append(
                head.forward(seq)
            )

        # ============================
        # Concatenate heads
        # ============================

        concat = np.concatenate(
            head_outputs,
            axis=1
        )

        # Save for output projection backward
        self.cache.append(concat)

        # ============================
        # Output projection
        # ============================

        out = concat @ self.Wo

        return out

    def backward(
        self,
        dout,
        learning_rate=1e-3
    ):

        # Must correspond to the same sequence
        # whose gradient is currently being processed.
        concat = self.cache.pop()

        # ============================
        # Output projection
        # ============================

        dWo = concat.T @ dout

        dconcat = dout @ self.Wo.T

        # ============================
        # Update Wo
        # ============================

        self.Wo -= (
            learning_rate * dWo
        )

        # ============================
        # Split gradient
        # ============================

        split_grads = np.split(
            dconcat,
            self.num_heads,
            axis=1
        )

        # ============================
        # Gradient to input
        # ============================

        dx = np.zeros(
            (
                concat.shape[0],
                self.embed_dim
            )
        )

        # Each head receives its own
        # portion of the concatenated gradient
        for grad, head in zip(
            split_grads,
            self.heads
        ):

            dx += head.backward(
                grad,
                learning_rate
            )

        return dx

In [98]:
class LayerNormalization:

    def __init__(self, eps=1e-5):
        self.eps = eps
        self.cache = []

    def forward(self, seq):

        if seq.ndim != 2:
            raise ValueError(
                f"LayerNormalization expected (sequence_length, embed_dim), "
                f"but received {seq.shape}"
            )

        mean = np.mean(
            seq,
            axis=1,
            keepdims=True
        )

        variance = np.mean(
            (seq - mean) ** 2,
            axis=1,
            keepdims=True
        )

        std = np.sqrt(
            variance + self.eps
        )

        x_hat = (seq - mean) / std

        self.cache.append(
            (
                seq,
                mean,
                variance,
                std,
                x_hat
            )
        )

        return x_hat

    def backward(self, dout):

        x, mean, variance, std, x_hat = self.cache.pop()

        D = x.shape[1]

        dx = (
            (1.0 / D)
            * (1.0 / std)
            * (
                D * dout
                - np.sum(
                    dout,
                    axis=1,
                    keepdims=True
                )
                - x_hat * np.sum(
                    dout * x_hat,
                    axis=1,
                    keepdims=True
                )
            )
        )

        return dx

In [99]:
class GELU:
    def __init__(self):

        self.cache = []

    def forward(self, x):

        t = np.sqrt(2/np.pi) * (
            x + 0.044715 * (x**3)
        )

        tanh = np.tanh(t)

        self.cache.append((x, tanh))

        return 0.5 * x * (1 + tanh)

    def backward(self, dout):

        x, tanh = self.cache.pop()

        c = np.sqrt(2/np.pi)

        dt_dx = c * (
            1 + 3 * 0.044715 * x**2
        )

        dy_dx = (
            0.5 * (1 + tanh)
            +
            0.5 * x * (1 - tanh**2) * dt_dx
        )

        dx = dout * dy_dx

        return dx

In [100]:
class FeedForward:
    def __init__(self, embed_dim):

        self.embed_dim = embed_dim
        self.hidden_dim = embed_dim * 4

        self.W1 = np.random.randn(
            embed_dim,
            self.hidden_dim
        ) * 0.02

        self.W2 = np.random.randn(
            self.hidden_dim,
            embed_dim
        ) * 0.02

        self.gelu = GELU()

        self.cache = []

    def forward(self, seq):

        linear1 = seq @ self.W1

        gelu_out = self.gelu.forward(linear1)

        output = gelu_out @ self.W2

        self.cache.append(
            (
                seq,
                gelu_out
            )
        )

        return output

    def backward(self, dout, learning_rate=1e-3):

        # IMPORTANT:
        # pop() means backward must process
        # sequences in reverse order.

        input, gelu_out = self.cache.pop()

        # -------------------------------
        # Linear 2
        # -------------------------------

        dW2 = gelu_out.T @ dout

        dgelu = dout @ self.W2.T

        # -------------------------------
        # GELU
        # -------------------------------

        dlinear1 = self.gelu.backward(dgelu)

        # -------------------------------
        # Linear 1
        # -------------------------------

        dW1 = input.T @ dlinear1

        dx = dlinear1 @ self.W1.T

        # -------------------------------
        # Update weights
        # -------------------------------

        self.W2 -= learning_rate * dW2

        self.W1 -= learning_rate * dW1

        return dx

In [101]:
class TransformerBlock:

    def __init__(
        self,
        embed_dim=256,
        num_heads=4,
        sequence_length=16
    ):

        self.ln1 = LayerNormalization()

        self.attention = MultiHeadAttention(
            embed_dim,
            num_heads,
            sequence_length
        )

        self.ln2 = LayerNormalization()

        self.ffn = FeedForward(embed_dim)

    def forward(self, batches):

        output = []

        for batch in batches:

            batch_out = []

            for seq in batch:

                # ============================
                # Attention
                # ============================

                residual = seq

                x = self.ln1.forward(seq)

                x = self.attention.forward(x)

                # Residual connection
                x = x + residual

                # ============================
                # Feed Forward
                # ============================

                residual = x

                y = self.ln2.forward(x)

                y = self.ffn.forward(y)

                # Residual connection
                y = y + residual

                batch_out.append(y)

            output.append(batch_out)

        return np.array(output)

    def backward(
        self,
        dout,
        learning_rate=1e-3
    ):

        output = []

        # IMPORTANT:
        #
        # Forward order:
        #   batch 0
        #      seq 0
        #      seq 1
        #      seq 2
        #
        # Backward must be:
        #   batch 0
        #      seq 2
        #      seq 1
        #      seq 0
        #
        # because all our layer caches use pop().

        for batch in reversed(dout):

            batch_grad = []

            for grad in reversed(batch):

                # ============================
                # Feed Forward residual
                # ============================

                residual_grad = grad

                grad = self.ffn.backward(
                    grad,
                    learning_rate
                )

                grad = self.ln2.backward(grad)

                # Residual gradient
                grad = grad + residual_grad

                # ============================
                # Attention residual
                # ============================

                residual_grad = grad

                grad = self.attention.backward(
                    grad,
                    learning_rate
                )

                grad = self.ln1.backward(grad)

                # Residual gradient
                grad = grad + residual_grad

                batch_grad.append(grad)

            # Restore original sequence order
            batch_grad.reverse()

            output.append(batch_grad)

        # Restore original batch order
        output.reverse()

        return np.array(output)

In [102]:
import numpy as np
import pickle


class GPT:

    def __init__(
        self,
        num_layers=1,
        embed_dim=256,
        num_heads=4,
        sequence_length=16,
        embedding_matrix=None
    ):

        self.num_layers = num_layers
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.sequence_length = sequence_length

        # ==========================================
        # Input embedding matrix
        # ==========================================

        if embedding_matrix is None:
            raise ValueError(
                "embedding_matrix must be provided"
            )

        self.embedding_matrix = embedding_matrix.copy()

        self.vocab_size = embedding_matrix.shape[0]

        # ==========================================
        # Separate output projection
        #
        # (embed_dim, vocab_size)
        # ==========================================

        self.W_out = (
            np.random.randn(
                embed_dim,
                self.vocab_size
            ) * 0.02
        )

        # ==========================================
        # Transformer blocks
        # ==========================================

        self.blocks = [
            TransformerBlock(
                embed_dim,
                num_heads,
                sequence_length
            )
            for _ in range(num_layers)
        ]

        # ==========================================
        # Final LayerNorm
        # ==========================================

        self.final_ln = LayerNormalization()

        # Final hidden representation cache
        self.x = None


    # ======================================================
    # FORWARD
    # ======================================================

    def forward(self, x):

        # x:
        # (batch,
        #  sequence_count,
        #  sequence_length,
        #  embed_dim)

        for block in self.blocks:
            x = block.forward(x)

        # ==========================================
        # Final LayerNorm
        # ==========================================

        final = []

        for batch in x:

            temp = []

            for seq in batch:

                temp.append(
                    self.final_ln.forward(seq)
                )

            final.append(temp)

        x = np.array(final)

        # Save for backward
        self.x = x

        # ==========================================
        # Vocabulary projection
        #
        # x:
        # (..., embed_dim)
        #
        # W_out:
        # (embed_dim, vocab_size)
        #
        # logits:
        # (..., vocab_size)
        # ==========================================

        logits = x @ self.W_out

        return logits


    # ======================================================
    # BACKWARD
    # ======================================================

    def backward(
        self,
        d_logits,
        learning_rate=1e-3
    ):

        # ==========================================
        # Flatten
        # ==========================================

        x_flat = self.x.reshape(
            -1,
            self.x.shape[-1]
        )

        d_logits_flat = d_logits.reshape(
            -1,
            d_logits.shape[-1]
        )

        # ==========================================
        # Output projection gradient
        #
        # logits = x @ W_out
        # ==========================================

        dW_out = (
            x_flat.T @ d_logits_flat
        )

        # Gradient into final LayerNorm
        dx = (
            d_logits @ self.W_out.T
        )

        # ==========================================
        # Update W_out
        # ==========================================

        self.W_out -= (
            learning_rate * dW_out
        )

        # ==========================================
        # Final LayerNorm backward
        # ==========================================

        final_dx = np.zeros_like(dx)

        # Reverse because LayerNorm cache uses pop()
        for b in range(
            dx.shape[0] - 1,
            -1,
            -1
        ):

            for s in range(
                dx.shape[1] - 1,
                -1,
                -1
            ):

                final_dx[b, s] = (
                    self.final_ln.backward(
                        dx[b, s]
                    )
                )

        dx = final_dx

        # ==========================================
        # Transformer blocks backward
        # ==========================================

        for block in reversed(self.blocks):

            dx = block.backward(
                dx,
                learning_rate
            )

        return dx


    # ======================================================
    # SAVE WEIGHTS
    # ======================================================

    def save_weights(
        self,
        filename="gpt_weights.pkl"
    ):

        data = {

            "num_layers":
                self.num_layers,

            "embed_dim":
                self.embed_dim,

            "num_heads":
                self.num_heads,

            "sequence_length":
                self.sequence_length,

            "vocab_size":
                self.vocab_size,

            # Input embeddings
            "embedding_matrix":
                self.embedding_matrix.copy(),

            # Separate output projection
            "W_out":
                self.W_out.copy(),
            "position_embeddings":
                position_embeddings.copy(),
            "blocks": []
        }

        # ==========================================
        # Transformer blocks
        # ==========================================

        for block in self.blocks:

            block_data = {

                "attention_Wo":
                    block.attention.Wo.copy(),

                "attention_heads": [],

                "ffn_W1":
                    block.ffn.W1.copy(),

                "ffn_W2":
                    block.ffn.W2.copy()
            }

            # ======================================
            # Attention heads
            # ======================================

            for head in block.attention.heads:

                head_data = {

                    "Wq":
                        head.Wq.copy(),

                    "Wk":
                        head.Wk.copy(),

                    "Wv":
                        head.Wv.copy()
                }

                block_data[
                    "attention_heads"
                ].append(head_data)

            data["blocks"].append(
                block_data
            )

        # ==========================================
        # Write checkpoint
        # ==========================================

        with open(filename, "wb") as f:
            pickle.dump(data, f)

        print(
            "Weights saved to:",
            filename
        )


    # ======================================================
    # LOAD WEIGHTS
    # ======================================================

    def load_weights(
        self,
        filename="gpt_weights.pkl"
    ):

        with open(filename, "rb") as f:
            data = pickle.load(f)

        # ==========================================
        # Architecture checks
        # ==========================================

        if data["num_layers"] != self.num_layers:
            raise ValueError(
                "Number of layers does not match"
            )

        if data["embed_dim"] != self.embed_dim:
            raise ValueError(
                "Embedding dimension does not match"
            )

        if data["num_heads"] != self.num_heads:
            raise ValueError(
                "Number of heads does not match"
            )

        if (
            data["sequence_length"]
            != self.sequence_length
        ):
            raise ValueError(
                "Sequence length does not match"
            )

        if (
            data.get(
                "vocab_size",
                data["embedding_matrix"].shape[0]
            )
            != self.vocab_size
        ):
            raise ValueError(
                "Vocabulary size does not match"
            )

        # ==========================================
        # Input embedding matrix
        # ==========================================

        if (
            data["embedding_matrix"].shape
            != self.embedding_matrix.shape
        ):
            raise ValueError(
                "Embedding matrix shape does not match"
            )

        self.embedding_matrix = (
            data["embedding_matrix"].copy()
        )

        # ==========================================
        # Output projection
        # ==========================================

        if "W_out" not in data:

            raise ValueError(
                "This checkpoint was created before "
                "the separate W_out output layer was added."
            )

        if (
            data["W_out"].shape
            != self.W_out.shape
        ):
            raise ValueError(
                "W_out shape does not match"
            )

        self.W_out = (
            data["W_out"].copy()
        )
        global position_embeddings

        if "position_embeddings" not in data:
            raise ValueError(
                "Checkpoint does not contain position_embeddings"
            )
        
        position_embeddings = (
            data["position_embeddings"].copy())

        # ==========================================
        # Transformer blocks
        # ==========================================

        if (
            len(data["blocks"])
            != len(self.blocks)
        ):
            raise ValueError(
                "Number of transformer blocks does not match"
            )

        for block, block_data in zip(
            self.blocks,
            data["blocks"]
        ):

            # --------------------------------------
            # Attention output projection
            # --------------------------------------

            if (
                block_data["attention_Wo"].shape
                != block.attention.Wo.shape
            ):
                raise ValueError(
                    "Attention Wo shape does not match"
                )

            block.attention.Wo = (
                block_data[
                    "attention_Wo"
                ].copy()
            )

            # --------------------------------------
            # Feed Forward
            # --------------------------------------

            if (
                block_data["ffn_W1"].shape
                != block.ffn.W1.shape
            ):
                raise ValueError(
                    "FFN W1 shape does not match"
                )

            if (
                block_data["ffn_W2"].shape
                != block.ffn.W2.shape
            ):
                raise ValueError(
                    "FFN W2 shape does not match"
                )

            block.ffn.W1 = (
                block_data[
                    "ffn_W1"
                ].copy()
            )

            block.ffn.W2 = (
                block_data[
                    "ffn_W2"
                ].copy()
            )

            # --------------------------------------
            # Attention heads
            # --------------------------------------

            if (
                len(
                    block_data[
                        "attention_heads"
                    ]
                )
                != len(
                    block.attention.heads
                )
            ):
                raise ValueError(
                    "Number of attention heads does not match"
                )

            for head, head_data in zip(
                block.attention.heads,
                block_data[
                    "attention_heads"
                ]
            ):

                head.Wq = (
                    head_data["Wq"].copy()
                )

                head.Wk = (
                    head_data["Wk"].copy()
                )

                head.Wv = (
                    head_data["Wv"].copy()
                )

        print(
            "Weights loaded from:",
            filename
        )

In [103]:
class CrossEntropyLoss:

    def __init__(self):
        self.loss = None

    def forward(self, logits, targets):
        """
        logits  : (batchs, batch_size, seq_len, vocab_size)
        targets : (batchs, batch_size, seq_len)
        """

        # Stable Softmax
        logits = logits - np.max(logits, axis=-1, keepdims=True)

        exp = np.exp(logits)
        probs = exp / np.sum(exp, axis=-1, keepdims=True)

        # Save probabilities (needed for backprop later)
        self.probs = probs
        self.targets = targets.astype(np.int64)

        # Pick probability of correct class
        correct_probs = np.take_along_axis(
            probs,
            targets[..., np.newaxis],
            axis=-1
        ).squeeze(-1)

        # Cross Entropy
        loss = -np.log(correct_probs + 1e-9)

        # Average over every token
        self.loss = np.mean(loss)

        return self.loss
    def backward(self):

        d_logits = self.probs.copy()
        B, S, L = self.targets.shape
        batch_idx = np.arange(B)[:, None, None]
        seq_idx   = np.arange(S)[None, :, None]
        token_idx = np.arange(L)[None, None, :]
        d_logits[batch_idx,
                 seq_idx,
                 token_idx,
                 self.targets] -= 1
    
        d_logits /= (B * S * L)
    
        return d_logits

In [104]:
gpt = GPT(
    num_layers=1,
    embed_dim=256,
    num_heads=4,
    sequence_length=16,
    embedding_matrix=embedding_matrix
)
criterion = CrossEntropyLoss()

In [105]:
gpt.load_weights(
    "gpt_stride4_epoch_15.pkl"
)

Weights loaded from: gpt_stride4_epoch_15.pkl


In [43]:
learning_rate = 1e-3

for epoch in range(14,15):

    total_loss = 0.0

    for batch_idx in range(len(embedding_inputs)):

        # ==========================================
        # Token IDs for this batch
        # ==========================================

        token_batch = np.asarray(
            embedding_inputs[batch_idx],
            dtype=np.int64
        )

        train_targets = np.asarray(
            target_outputs[batch_idx],
            dtype=np.int64
        )

        # ==========================================
        # Token IDs -> CURRENT GPT embeddings
        # ==========================================

        train_inputs = gpt.embedding_matrix[
            token_batch
        ]

        # Add positional embeddings
        train_inputs = (
            train_inputs
            + position_embeddings
        )

        # Add outer dimensions
        train_inputs = train_inputs[
            np.newaxis, :, :, :
        ]

        train_targets = train_targets[
            np.newaxis, :, :
        ]

        # ==========================================
        # Forward
        # ==========================================

        logits = gpt.forward(
            train_inputs
        )

        # ==========================================
        # Loss
        # ==========================================

        loss = criterion.forward(
            logits,
            train_targets
        )

        total_loss += loss

        # ==========================================
        # Backward
        # ==========================================

        d_logits = criterion.backward()

        gpt.backward(
            d_logits,
            learning_rate=learning_rate
        )

        # ==========================================
        # Progress
        # ==========================================

        if batch_idx % 500 == 0:

            print(
                "Epoch:",
                epoch + 1,
                "Batch:",
                batch_idx,
                "Loss:",
                loss
            )

    average_loss = (
        total_loss
        / len(embedding_inputs)
    )

    print()
    print(
        "Epoch:",
        epoch + 1,
        "Average Loss:",
        average_loss
    )
    print()

    # ==========================================
    # Save complete checkpoint
    # ==========================================

    gpt.save_weights(
        f"gpt_stride4_epoch_{epoch + 1}.pkl"
    )

Epoch: 15 Batch: 0 Loss: 3.1876890930579096
Epoch: 15 Batch: 500 Loss: 2.55666479555335
Epoch: 15 Batch: 1000 Loss: 2.324841164500576
Epoch: 15 Batch: 1500 Loss: 2.4602554259054092
Epoch: 15 Batch: 2000 Loss: 2.298870922725553
Epoch: 15 Batch: 2500 Loss: 1.543537749975394
Epoch: 15 Batch: 3000 Loss: 2.4798223713897727
Epoch: 15 Batch: 3500 Loss: 2.0609979519918795
Epoch: 15 Batch: 4000 Loss: 3.0680974704254305
Epoch: 15 Batch: 4500 Loss: 1.9626999986390645
Epoch: 15 Batch: 5000 Loss: 2.1350227060608016
Epoch: 15 Batch: 5500 Loss: 2.119892058391667
Epoch: 15 Batch: 6000 Loss: 2.807103531091699
Epoch: 15 Batch: 6500 Loss: 2.5338463047387267
Epoch: 15 Batch: 7000 Loss: 2.111965748646547
Epoch: 15 Batch: 7500 Loss: 1.6913801938624344
Epoch: 15 Batch: 8000 Loss: 1.8681669133190464
Epoch: 15 Batch: 8500 Loss: 2.1557960114127415
Epoch: 15 Batch: 9000 Loss: 1.7104203988666997
Epoch: 15 Batch: 9500 Loss: 1.9938189519122684
Epoch: 15 Batch: 10000 Loss: 2.6116130663712873
Epoch: 15 Batch: 10500 L

In [41]:
space_id = tokenizer.str_to_int[" "]

correct = 0
total = 0

non_space_correct = 0
non_space_total = 0

space_predictions = 0


for batch_idx in range(len(vector_inputs)):

    # ==========================================
    # Prepare input
    # ==========================================

    train_inputs = np.asarray(
        vector_inputs[batch_idx],
        dtype=np.float64
    )

    train_targets = np.asarray(
        target_outputs[batch_idx],
        dtype=np.int64
    )

    # Add positional embeddings
    train_inputs = (
        train_inputs
        + position_embeddings
    )

    # Add outer dimensions
    train_inputs = train_inputs[
        np.newaxis, :, :, :
    ]

    train_targets = train_targets[
        np.newaxis, :, :
    ]

    # ==========================================
    # Forward
    # ==========================================

    logits = gpt.forward(
        train_inputs
    )

    predictions = np.argmax(
        logits,
        axis=-1
    )

    # ==========================================
    # 2. Overall accuracy
    # ==========================================

    correct += np.sum(
        predictions == train_targets
    )

    total += train_targets.size


    # ==========================================
    # 3. Non-space accuracy
    # ==========================================

    non_space_mask = (
        train_targets != space_id
    )

    non_space_correct += np.sum(
        (predictions == train_targets)
        & non_space_mask
    )

    non_space_total += np.sum(
        non_space_mask
    )


    # ==========================================
    # 4. Space prediction percentage
    # ==========================================

    space_predictions += np.sum(
        predictions == space_id
    )


    # ==========================================
    # Clear inference caches
    # ==========================================

    gpt.final_ln.cache.clear()

    for block in gpt.blocks:

        block.ln1.cache.clear()
        block.ln2.cache.clear()

        block.attention.cache.clear()

        for head in block.attention.heads:
            head.cache.clear()

        block.ffn.cache.clear()
        block.ffn.gelu.cache.clear()


# ==============================================
# Final metrics
# ==============================================

overall_accuracy = (
    correct / total
)

non_space_accuracy = (
    non_space_correct / non_space_total
)

space_prediction_percentage = (
    space_predictions / total
)


print(
    "Overall Accuracy:",
    overall_accuracy
)

print(
    "Non-Space Accuracy:",
    non_space_accuracy
)

print(
    "Space Prediction Percentage:",
    space_prediction_percentage
)

Overall Accuracy: 0.17918556992634663
Non-Space Accuracy: 0.006598004657747963
Space Prediction Percentage: 0.5662305051468631


In [106]:
correct = 0
total = 0

non_space_correct = 0
non_space_total = 0

space_predictions = 0

space_id = tokenizer.str_to_int[" "]

for batch_idx in range(100):

    token_batch = np.asarray(
        embedding_inputs[batch_idx],
        dtype=np.int64
    )

    inputs = gpt.embedding_matrix[
        token_batch
    ]

    inputs = inputs + position_embeddings

    inputs = inputs[
        np.newaxis, :, :, :
    ]

    targets = np.asarray(
        target_outputs[batch_idx],
        dtype=np.int64
    )[np.newaxis, :, :]

    logits = gpt.forward(inputs)

    predictions = np.argmax(
        logits,
        axis=-1
    )

    # Overall
    correct += np.sum(
        predictions == targets
    )

    total += targets.size

    # Non-space
    mask = targets != space_id

    non_space_correct += np.sum(
        (predictions == targets) & mask
    )

    non_space_total += np.sum(mask)

    # Space predictions
    space_predictions += np.sum(
        predictions == space_id
    )

    # Clear caches
    gpt.final_ln.cache.clear()

    for block in gpt.blocks:

        block.ln1.cache.clear()
        block.ln2.cache.clear()

        block.attention.cache.clear()

        for head in block.attention.heads:
            head.cache.clear()

        block.ffn.cache.clear()
        block.ffn.gelu.cache.clear()


print("Overall Accuracy:", correct / total)
print("Non-Space Accuracy:", non_space_correct / non_space_total)
print("Space Prediction Percentage:", space_predictions / total)

Overall Accuracy: 0.412890625
Non-Space Accuracy: 0.24350892189336468
Space Prediction Percentage: 0.34703125


In [107]:
def generate(
    gpt,
    tokenizer,
    prompt,
    max_new_tokens=20,
    temperature=0.8,
    do_sample=True,
    top_k=10
):

    token_ids = tokenizer.encode(prompt)

    for _ in range(max_new_tokens):

        # ==========================================
        # Keep only model context
        # ==========================================

        context_ids = token_ids[
            -gpt.sequence_length:
        ]

        # ==========================================
        # Token IDs -> embeddings
        # ==========================================

        x = gpt.embedding_matrix[
            np.asarray(
                context_ids,
                dtype=np.int64
            )
        ]

        # ==========================================
        # Add positional embeddings
        # ==========================================

        x = (
            x
            + position_embeddings[
                :len(context_ids)
            ]
        )

        # ==========================================
        # Add outer dimensions
        # ==========================================

        x = x[
            np.newaxis,
            np.newaxis,
            :,
            :
        ]

        # ==========================================
        # Forward
        # ==========================================

        logits = gpt.forward(x)

        next_logits = (
            logits[0, 0, -1].copy()
        )

        # ==========================================
        # Never generate <unk>
        # ==========================================

        unk_id = tokenizer.str_to_int["<unk>"]

        next_logits[unk_id] = -np.inf

        # ==========================================
        # Greedy decoding
        # ==========================================

        if not do_sample:

            next_token = np.argmax(
                next_logits
            )

        # ==========================================
        # Top-k Sampling
        # ==========================================

        else:

            if temperature <= 0:
                raise ValueError(
                    "temperature must be greater than 0"
                )

            # ------------------------------
            # Select top-k tokens only
            # ------------------------------

            k = min(
                top_k,
                len(next_logits)
            )

            top_ids = np.argsort(
                next_logits
            )[-k:]

            top_logits = (
                next_logits[top_ids]
                / temperature
            )

            # ------------------------------
            # Stable softmax
            # ------------------------------

            top_logits -= np.max(
                top_logits
            )

            probs = np.exp(
                top_logits
            )

            probs /= np.sum(
                probs
            )

            # ------------------------------
            # Sample from top-k only
            # ------------------------------

            next_token = np.random.choice(
                top_ids,
                p=probs
            )

        # ==========================================
        # Append generated token
        # ==========================================

        token_ids.append(
            int(next_token)
        )

        # ==========================================
        # Clear inference caches
        # ==========================================

        gpt.final_ln.cache.clear()

        for block in gpt.blocks:

            block.ln1.cache.clear()
            block.ln2.cache.clear()

            block.attention.cache.clear()

            for head in block.attention.heads:
                head.cache.clear()

            block.ffn.cache.clear()
            block.ffn.gelu.cache.clear()

    # ==============================================
    # Decode
    # ==========================================

    return tokenizer.decode(token_ids)

In [111]:
prompts = [
    "Sherlock Holmes was ",
    "Holmes said ",
    "The man ",
    "I was ",
    "It was "
]

for prompt in prompts:

    generated = generate(
        gpt,
        tokenizer,
        prompt,
        max_new_tokens=30,
        temperature=0.8,
        do_sample=False,
        top_k=10
    )

    print("Prompt:", repr(prompt))
    print("Generated:", repr(generated))
    print()

Prompt: 'Sherlock Holmes was '
Generated: 'Sherlock Holmes was a small came to be a part of the disappeared'

Prompt: 'Holmes said '
Generated: 'Holmes said he. “I have been cleared to the day in the day of'

Prompt: 'The man '
Generated: 'The man who was a servant of the disappeared to the cord'

Prompt: 'I was '
Generated: 'I was a perhappened the disappeared to the day of the '

Prompt: 'It was '
Generated: 'It was a person the disappeared to the day of the disap'



In [109]:
prompts = [
    "The sun was ",
    "A young boy ",
    "My hobby is writing poems ",
    "In the morning ",
    "The old house ",
    "He looked at ",
    "There was a ",
    "I could not ",
    "The woman said ",
    "After a long time "
]

for prompt in prompts:

    generated = generate(
        gpt,
        tokenizer,
        prompt,
        max_new_tokens=40,
        temperature=0.8,
        do_sample=True,
        top_k=10
    )

    print("Prompt:", repr(prompt))
    print("Generated:", repr(generated))
    print()

Prompt: 'The sun was '
Generated: 'The sun was a morning, and the\nfirst to the married, and he came bright of '

Prompt: 'A young boy '
Generated: 'A young boy of the maker Street, and I could have been clear, but he could have '

Prompt: 'My hobby is writing poems '
Generated: 'My hobby is writing poems of the prison of his counces of the castle was down the\nday befo'

Prompt: 'In the morning '
Generated: 'In the morning at the day what I was not know that we have the disper on the house,'

Prompt: 'The old house '
Generated: 'The old house of the corner of a parent in the barning in the matter clearly to'

Prompt: 'He looked at '
Generated: 'He looked at the down. He sprang in the house, and\nso plish, and I kne'

Prompt: 'There was a '
Generated: 'There was a corner that you must be a pill\npast, and I must have the disrem'

Prompt: 'I could not '
Generated: 'I could not think that the scend in a scried, and I made became with the '

Prompt: 'The woman said '
Generated: 'The woman

In [112]:
print("Prompt IDs:")
print(tokenizer.encode("Sherlock Holmes was "))

print("\nPosition embeddings:")
print(position_embeddings[0, :10])

print("\nGPT embedding:")
print(gpt.embedding_matrix[0, :10])

print("\nW_out:")
print(gpt.W_out[0, :10])

Prompt IDs:
[45, 115, 139, 82, 2, 33, 2, 212, 2]

Position embeddings:
[ 0.03556366  0.01902241 -0.01873993 -0.03144063 -0.00633959  0.00131123
 -0.00897838 -0.00302989  0.01491648 -0.01722548]

GPT embedding:
[-0.02575459  0.0032443  -0.01526989 -0.01332201 -0.01227695  0.00162816
 -0.01219321  0.02644115 -0.01564262  0.00057349]

W_out:
[-0.011323    0.10980164  0.01349074 -0.01349675  0.01167712  0.00637805
 -0.01267316  0.01971316 -0.04500713 -0.02295188]
